<a href="https://colab.research.google.com/github/MohammedAyman07/curriculum-rag-research/blob/main/notebooks/02_tfidf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CurriculumRAG — TF-IDF Retrieval

## Objective
Implement keyword-based document retrieval using TF-IDF.

## Research Method
1. Load the educational passage collection.
2. Convert passages into TF-IDF vectors.
3. Convert user questions into TF-IDF vectors.
4. Calculate similarity scores.
5. Retrieve the five most relevant passages.

## Evaluation
Recall@5, Precision@5, MRR and query latency.

In [3]:

# CurriculumRAG - TF-IDF environment

import numpy as np
import pandas as pd
import time
import os

from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("TF-IDF environment ready!")


TF-IDF environment ready!


In [4]:

# Load datasets from GitHub

BASE_URL = (
    "https://raw.githubusercontent.com/"
    "MohammedAyman07/curriculum-rag-research/main/data/"
)

passages_df = pd.read_csv(BASE_URL + "passages.csv")
queries_df = pd.read_csv(BASE_URL + "queries.csv")

assert len(passages_df) == 40
assert len(queries_df) == 60

documents = passages_df["text"].fillna("").astype(str).tolist()
document_ids = passages_df["doc_id"].astype(str).tolist()

print("Passages loaded:", len(documents))
print("Questions loaded:", len(queries_df))
print("Dataset ready for TF-IDF!")


Passages loaded: 40
Questions loaded: 60
Dataset ready for TF-IDF!


In [5]:

# Build the TF-IDF document index

vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True,
    token_pattern=r"(?u)\b\w+\b"
)

index_start = time.perf_counter()

document_matrix = vectorizer.fit_transform(documents)

index_time = time.perf_counter() - index_start

print("TF-IDF index created successfully!")
print("Document matrix shape:", document_matrix.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))
print(f"Index creation time: {index_time:.4f} seconds")


TF-IDF index created successfully!
Document matrix shape: (40, 2195)
Vocabulary size: 2195
Index creation time: 0.0468 seconds


In [6]:

# Retrieve the top-k most relevant passages

def retrieve_tfidf(question, top_k=5):

    start = time.perf_counter()

    query_vector = vectorizer.transform([question])

    similarity_scores = cosine_similarity(
        query_vector,
        document_matrix
    ).flatten()

    # Stable ordering makes ties reproducible
    ranked_indices = np.argsort(
        -similarity_scores,
        kind="stable"
    )[:top_k]

    results = []

    for index in ranked_indices:
        results.append({
            "doc_id": document_ids[index],
            "score": float(similarity_scores[index]),
            "topic": passages_df.iloc[index]["topic"],
            "text": documents[index]
        })

    latency_ms = (time.perf_counter() - start) * 1000

    return results, latency_ms


# Test with one question
sample_question = "How does binary search reduce the search space?"

sample_results, sample_latency = retrieve_tfidf(sample_question)

print("Question:", sample_question)
print(f"Query latency: {sample_latency:.4f} ms\n")

for rank, result in enumerate(sample_results, start=1):
    print(f"Rank {rank}: {result['doc_id']}")
    print("Topic:", result["topic"])
    print("Similarity:", round(result["score"], 4))
    print()


Question: How does binary search reduce the search space?
Query latency: 4.2267 ms

Rank 1: P033
Topic: Binary Search
Similarity: 0.2253

Rank 2: P014
Topic: Binary Search Trees
Similarity: 0.1128

Rank 3: P015
Topic: AVL Trees
Similarity: 0.1028

Rank 4: P002
Topic: Space Complexity
Similarity: 0.0501

Rank 5: P034
Topic: Hashing
Similarity: 0.0479



In [7]:

# Evaluation metrics

def calculate_metrics(retrieved_ids, relevant_ids, k=5):

    retrieved = retrieved_ids[:k]
    relevant = set(relevant_ids)

    hits = set(retrieved) & relevant

    precision = len(hits) / k
    recall = len(hits) / len(relevant)

    reciprocal_rank = 0.0

    for rank, doc_id in enumerate(retrieved, start=1):
        if doc_id in relevant:
            reciprocal_rank = 1 / rank
            break

    return precision, recall, reciprocal_rank


# Evaluate development questions only

dev_queries = queries_df[
    queries_df["split"] == "dev"
].copy()

evaluation_records = []

for _, row in dev_queries.iterrows():

    results, _ = retrieve_tfidf(row["question"], top_k=5)

    retrieved_ids = [
        result["doc_id"] for result in results
    ]

    relevant_ids = str(
        row["relevant_doc_ids"]
    ).split("|")

    precision, recall, mrr = calculate_metrics(
        retrieved_ids,
        relevant_ids,
        k=5
    )

    # Repeat timing to reduce one-off measurement noise
    latency_samples = [
        retrieve_tfidf(row["question"], top_k=5)[1]
        for _ in range(5)
    ]

    evaluation_records.append({
        "query_id": row["query_id"],
        "question": row["question"],
        "relevant_doc_ids": row["relevant_doc_ids"],
        "retrieved_doc_ids": "|".join(retrieved_ids),
        "precision_at_5": precision,
        "recall_at_5": recall,
        "mrr": mrr,
        "median_query_latency_ms": float(
            np.median(latency_samples)
        )
    })


dev_results_df = pd.DataFrame(evaluation_records)

print("TF-IDF development evaluation completed!")
display(dev_results_df)


TF-IDF development evaluation completed!


,query_id,question,relevant_doc_ids,retrieved_doc_ids,precision_at_5,recall_at_5,mrr,median_query_latency_ms
0,Q001,How does an algorithm's running time change as...,P001,P001|P002|P023|P037|P020,0.2,1.0,1.000000,1.070479
1,Q004,Why can an array element be accessed directly ...,P004,P004|P003|P034|P026|P028,0.2,1.0,1.000000,0.968150
2,Q006,Which linked list supports traversal in both d...,P006,P006|P007|P005|P003|P036,0.2,1.0,1.000000,0.891340
3,Q008,Which principle explains why the latest stack ...,P008,P008|P010|P020|P013|P026,0.2,1.0,1.000000,0.904980
4,Q010,Where does a queue insert new elements and rem...,P010,P040|P025|P010|P019|P032,0.2,1.0,0.333333,0.913589
5,Q013,How many children can a node have in a binary ...,P013,P014|P013|P016|P015|P030,0.2,1.0,0.500000,0.901759
6,Q015,What balance condition does an AVL tree mainta...,P015,P015|P005|P013|P023|P014,0.2,1.0,1.000000,0.881269
7,Q018,"What are the main components of a graph, and h...",P017,P017|P021|P014|P012|P008,0.2,1.0,1.000000,0.893530
8,Q020,Which graph traversal visits nearby vertices b...,P019,P016|P019|P020|P013|P017,0.2,1.0,0.500000,0.924259
9,Q022,How does DFS explore a graph and then return t...,P020,P019|P018|P023|P020|P017,0.2,1.0,0.250000,0.915930


In [8]:

# Calculate average development metrics

summary = {
    "Method": "TF-IDF",
    "Precision@5": dev_results_df["precision_at_5"].mean(),
    "Recall@5": dev_results_df["recall_at_5"].mean(),
    "MRR": dev_results_df["mrr"].mean(),
    "Median Query Latency (ms)": (
        dev_results_df["median_query_latency_ms"].median()
    ),
    "Index Creation Time (seconds)": index_time
}

summary_df = pd.DataFrame([summary])

display(summary_df)

# Save development results
RESULTS_DIR = Path("/content/curriculum-rag-results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

dev_results_df.to_csv(
    RESULTS_DIR / "tfidf_dev_predictions.csv",
    index=False
)

summary_df.to_csv(
    RESULTS_DIR / "tfidf_dev_metrics.csv",
    index=False
)

print("TF-IDF development results saved!")


,Method,Precision@5,Recall@5,MRR,Median Query Latency (ms),Index Creation Time (seconds)
0,TF-IDF,0.2,1.0,0.805556,0.90498,0.046774


TF-IDF development results saved!
